<a href="https://colab.research.google.com/github/IgorAkO/Simulative_ML/blob/main/1_Basic%20Python/S_ML_1_14_1_Logs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1 кейс**

**Работа с логами**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [15]:
!wget https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log

--2026-03-05 04:32:27--  https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.github.com (gist.github.com)... 140.82.114.3
Connecting to gist.github.com (gist.github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log [following]
--2026-03-05 04:32:27--  https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 459418 (449K) [text/plain]
Saving to: ‘auto_purchase.log.1’

auto_purchase.log.1 100%[===================>] 448.65K  --.-KB/s    in 0.005s  

2026-03-05 04:32:27 (93

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [ ]:
with open('auto_purchase.log', 'r') as f:
  lines = f.readlines()

for line in lines[410:415]:
  print(line)

INFO  | 2022-11-29 00:01:02,399  | file: demon_auto_purchase_bundle.py |  line: 61 | [demon] Обновляем подписку пользователю id: 5014, payment_method_id: 2af064ab

INFO  | 2022-11-29 00:01:04,386  | file: demon_auto_purchase_bundle.py |  line: 61 | [demon] Обновляем подписку пользователю id: 5019, payment_method_id: 2af0ffe6

INFO  | 2022-11-30 00:01:02,216  | file: demon_auto_purchase_bundle.py |  line: 44 | [demon] Поверяем истекающие премиум-пакеты

INFO  | 2022-11-30 00:01:02,245  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-30, количество людей с автопродлением подписки: 0

INFO  | 2022-11-30 00:01:02,251  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы



### **Решения**

#### **Задача 1**

Ваша задача написать функцию `count_success_and_failure`, которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.

**Решение**

Напишите свое решение ниже

In [7]:
def count_success_and_failure(file_path):
  with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

  logs = [[line_p.strip() for line_p in line.split('|')] for line in lines]

  # Инф из описания логов logs[i][4]
  s = [logs[i] for i in range(len(logs))
    if 'Обновляем подписку пользователю id:' in logs[i][4]
    and (i == len(logs)-1 or 'ошибка при списании' not in logs[i+1][4])]
  #print(s)

  er = [logs[i+1] for i in range(len(logs)-1)
    if 'ошибка при списании' in logs[i+1][4]
    and 'Обновляем подписку пользователю id:' in logs[i][4]]
  #print(er)

  return len(s), len(er)

print(count_success_and_failure('auto_purchase.log'))

(1034, 186)


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [8]:
res = count_success_and_failure('auto_purchase.log')

try:
    assert res == (1034, 186)
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача написать функцию `auto_renewal_sub`, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с автопродлением подписки. Мы хотим посмотреть на изменение этого показателя в динамике: посчитайте сглаженные значения с помощью метода скользящего среднего и метода медианного сглаживания.  

**Примечание:** При сглаживании берем все предыдущие значения, включая текущее, будущие значения не берем. Если в один день наблюдаем несколько записей об автопродлении - берем максимальное из имеющихся число клиентов с подпиской.

Функция должна записать в файл `auto_renewal_sub.txt` два списка, предварив их соответствущими обозначениями:

`Среднее: [2.0, 1.0, 0.67...]`

`Медиана: [2, 2, 0...]`

**Решение**

Напишите свое решение ниже

In [17]:
import statistics

def auto_renewal_sub(log_file_path):
  with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

  d_auto_ren = {}
  for line in lines:
    desc = line.split('|')[4].strip()
    if desc.startswith('[demon] Cегодня') == True:
      date = desc[16:26]
      num = int(desc[72:])
      if date in d_auto_ren.keys():
        d_auto_ren[date] = max(d_auto_ren[date], num)
      else:
        d_auto_ren[date] = num

  vals = list(d_auto_ren.values())
  mean_list = list(map(
      lambda i: round(sum(vals[:i+1])/len(vals[:i+1]), 2), range(len(vals))
      ))
  median_list = [int(statistics.median(vals[:i+1])) for i in range(len(vals))]

  # Записываем результат в файл 'auto_renewal_sub.txt'
  with open('auto_renewal_sub.txt', 'w') as f:
    f.write("Среднее: {}\n".format(mean_list))
    f.write("Медиана: {}\n".format(median_list))

  return mean_list, median_list, vals


mean_list, median_list, auto_ren = auto_renewal_sub('auto_purchase.log')
print(mean_list)
print(median_list)
print(auto_ren)

[1.0, 0.5, 0.33, 0.25, 0.2, 0.17, 0.14, 0.12, 0.11, 0.1, 0.09, 0.08, 0.08, 0.07, 0.07, 0.06, 0.06, 0.06, 0.05, 0.05, 0.05, 0.05, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.03, 0.03, 0.06, 0.09, 0.09, 0.09, 0.09, 0.08, 0.08, 0.08, 0.08, 0.07, 0.07, 0.07, 0.12, 0.14, 0.13, 0.13, 0.13, 0.15, 0.16, 0.16, 0.16, 0.15, 0.15, 0.15, 0.15, 0.14, 0.16, 0.17, 0.17, 0.17, 0.2, 0.21, 0.21, 0.2, 0.22, 0.21, 0.21, 0.21, 0.2, 0.2, 0.2, 0.19, 0.21, 0.2, 0.2, 0.2, 0.19, 0.21, 0.22, 0.21, 0.22, 0.22, 0.22, 0.21, 0.21, 0.21, 0.22, 0.22, 0.21, 0.22, 0.24, 0.24, 0.25, 0.24, 0.24, 0.25, 0.25, 0.24, 0.24, 0.25, 0.25, 0.25, 0.24, 0.24, 0.24, 0.24, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.25, 0.25, 0.25, 0.25, 0.25, 0.24, 0.25, 0.27, 0.27, 0.27, 0.26, 0.27, 0.28, 0.28, 0.28, 0.28, 0.29, 0.29, 0.3, 0.3, 0.3, 0.3, 0.29, 0.29, 0.3, 0.31, 0.31, 0.31, 0.31, 0.31, 0.33, 0.33, 0.34, 0.34, 0.34, 0.34, 0.

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [18]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt

import pandas as pd

user_answer = pd.read_csv('auto_renewal_sub.txt')
correct_answer = pd.read_csv('cor_auto_renewal.txt')

--2026-03-05 04:34:25--  https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt
Resolving gist.github.com (gist.github.com)... 140.82.112.3
Connecting to gist.github.com (gist.github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt [following]
--2026-03-05 04:34:25--  https://gist.githubusercontent.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2431 (2.4K) [text/plain]
Saving to: ‘cor_auto_renewal.txt.1’

cor_auto_renewal.tx 100%[===================>]   2.37K  --.-KB/s    in 0s      

2026-03-05 04

In [19]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!


#### **Задача 3**

Напишите функцию `sub_renewal_by_day`, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл `weekdays.txt` аналитическую записку в формате:

**`Количество обновлений подписки по дням недели:`**

**`Понедельник: 6`**

**`Вторник: 7`**

**`Среда: 8`**

**`...`**

**Решение**

Напишите свое решение ниже

In [35]:
from datetime import datetime

def sub_renewal_by_day(file_path):
  with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

  logs = [[line_p.strip() for line_p in line.split('|')] for line in lines]

  # Инф из описания логов logs[i][1] - дата, logs[i][1] - удачное обновление
  s = [logs[i][1] for i in range(len(logs)) if 'Обновляем подписку пользователю id:' in logs[i][4]
    and (i == len(logs)-1 or 'ошибка при списании' not in logs[i+1][4])]

  week_list = [0, 0, 0, 0, 0, 0, 0]
  for week_day in s:
    w = datetime.strptime(week_day, '%Y-%m-%d %H:%M:%S,%f').weekday()
    week_list[w] += 1


  #w_day = datetime.strptime(line.split('|')[1][1:11], "%Y-%m-%d").weekday()
  w_day = ['Понедельник', 'Вторник', 'Среда', 'Четверг', 'Пятница', 'Суббота',
           'Воскресенье']

  with open('weekdays.txt', 'w') as f:
    print('Количество обновлений подписки по дням недели:', file=f)
    for i in range(7):
            print(f'{w_day[i]}: {week_list[i]}', file=f)

  return week_list

week_list = sub_renewal_by_day('auto_purchase.log')
print(week_list)

[136, 144, 162, 169, 145, 135, 143]


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [36]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt

import pandas as pd

user_answer = pd.read_csv('weekdays.txt')
correct_answer = pd.read_csv('cor_weekdays.txt')

--2026-03-05 05:18:36--  https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt
Resolving gist.github.com (gist.github.com)... 140.82.113.4
Connecting to gist.github.com (gist.github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt [following]
--2026-03-05 05:18:36--  https://gist.githubusercontent.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 238 [text/plain]
Saving to: ‘cor_weekdays.txt’

cor_weekdays.txt    100%[===================>]     238  --.-KB/s    in 0s      

2026-03-05 05:18:36 (5.56 MB/s) - ‘cor_

In [37]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
